In [6]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.3 MB/s eta 0:00:00


In [3]:
from pathlib import Path
from datetime import datetime

import json
import platform
import random
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics

from IPython.display import display
from PIL import Image
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [11]:
PROJECT_ROOT = Path.cwd()

DRIVE_DATASETS_DIR = Path("/content/drive/MyDrive/Yolo_road2022_dataset")
DATASET_ROOT = DRIVE_DATASETS_DIR / "RDD2022_China_MotorBike"

TRAIN_IMAGES_DIR = DATASET_ROOT / "train" / "images"
TRAIN_LABELS_DIR = DATASET_ROOT / "train" / "labels"

VAL_IMAGES_DIR = DATASET_ROOT / "val" / "images"
VAL_LABELS_DIR = DATASET_ROOT / "val" / "labels"

CONFIGS_DIR = DATASET_ROOT / "configs"
REPORTS_DIR = DATASET_ROOT / "reports"

DATA_YAML_PATH = CONFIGS_DIR / "rdd2022_china_motorbike.yaml"

SPLIT_REPORT_PATH = REPORTS_DIR / "rdd2022_china_motorbike_split_manifest.json"

RUNS_ROOT = DATASET_ROOT / "runs"

print("Data YAML :", DATA_YAML_PATH.resolve())
print("Runs root :", RUNS_ROOT.resolve())

Data YAML : /content/drive/MyDrive/Yolo_road2022_dataset/RDD2022_China_MotorBike/configs/rdd2022_china_motorbike.yaml
Runs root : /content/drive/MyDrive/Yolo_road2022_dataset/RDD2022_China_MotorBike/runs


In [ ]:
IMAGE_SUFFIXES = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
}

train_images = sorted(
    path
    for path in TRAIN_IMAGES_DIR.iterdir()
    if path.suffix.lower() in IMAGE_SUFFIXES
)

val_images = sorted(
    path
    for path in VAL_IMAGES_DIR.iterdir()
    if path.suffix.lower() in IMAGE_SUFFIXES
)

train_labels = sorted(TRAIN_LABELS_DIR.glob("*.txt"))
val_labels = sorted(VAL_LABELS_DIR.glob("*.txt"))

print("Training images  :", len(train_images))
print("Training labels  :", len(train_labels))
print("Validation images:", len(val_images))
print("Validation labels:", len(val_labels))

Training images  : 1547
Training labels  : 1547
Validation images: 387
Validation labels: 387


**Training configuration**

In [10]:
MODEL_ID = "yolo26n.pt"
IMAGE_SIZE = 640
EPOCHS = 50
PATIENCE = 15
TRAIN_BATCH = 15
VALIDATION_BATCH = 8
SEED = 42
WORKERS = 0
RUN_NAME = "yolo26n_rdd2022_baseline"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [12]:
training_configuration = {
    "model" : MODEL_ID,
    "data" : str(DATA_YAML_PATH.resolve()),
    "epochs" : EPOCHS,
    "imgsz" : IMAGE_SIZE,
    "batch" : TRAIN_BATCH,
    "device" : DEVICE,
    "workers" : WORKERS,
    "patience" : PATIENCE,
    "optimizer" : "auto",
    "warmup_epochs" : 1.0,
    "close_mosaic" : 10,
    "amp" : True,
    "cache" : False,
    "seed" : SEED,
    "deterministic" : True,
    "val" : True,
    "plots" : True,
    "save" : True,
    "save_period" : 5,
    "project" : str(RUNS_ROOT.resolve()),
    "name" : RUN_NAME,
    "exit_ok" : False,
    "verbose" : True
}

for key, value in training_configuration.items():
    print(f"{key:<18}: {value}")

model             : yolo26n.pt
data              : /content/drive/MyDrive/Yolo_road2022_dataset/RDD2022_China_MotorBike/configs/rdd2022_china_motorbike.yaml
epochs            : 50
imgsz             : 640
batch             : 15
device            : cuda
workers           : 0
patience          : 15
optimizer         : auto
warmup_epochs     : 1.0
close_mosaic      : 10
amp               : True
cache             : False
seed              : 42
deterministic     : True
val               : True
plots             : True
save              : True
save_period       : 5
project           : /content/drive/MyDrive/Yolo_road2022_dataset/RDD2022_China_MotorBike/runs
name              : yolo26n_rdd2022_baseline
exit_ok           : False
verbose           : True
